# 02 — BOCPD Across Repetitions

Bayesian Online Change-Point Detection on log bandpower **across repetitions** (DATASET_GUIDE §8B). For a fixed condition, the sequence of 80 test reps is the time series; BOCPD flags regime shifts.

Reference: Adams & MacKay (2007); `refrences/deep-research-report.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import norm

def logsumexp(x):
    m = np.max(x)
    return m + np.log(np.sum(np.exp(x - m)))

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else (Path.cwd() / ".." / "..").resolve()
FEATURES_DIR = ROOT / "artifacts" / "data" / "features__log_bandpower__2026-02-27"
ARTIFACTS_FIG = ROOT / "artifacts" / "figures" / "bocpd"
ARTIFACTS_FIG.mkdir(parents=True, exist_ok=True)

## BOCPD implementation (Gaussian, unknown mean)

Constant hazard H(r)=h. Predictive: N(μ_n, σ_n² + σ_x²). Log-space for stability.

In [ ]:
def bocpd_gaussian(x, hazard=0.02, mean0=None, var0=1.0, varx=None):
    """
    BOCPD for scalar Gaussian: x|μ~N(μ,varx), μ~N(mean0,var0).
    Returns: R (run-length posterior), cp_prob (changepoint prob per t).
    """
    T = len(x)
    if mean0 is None:
        mean0 = np.mean(x)
    if varx is None:
        varx = np.var(x) + 1e-6
    prec0 = 1.0 / var0
    precx = 1.0 / varx
    log_H = np.log(hazard)
    log_1mH = np.log(1 - hazard)
    R = np.zeros((T + 1, T + 1))
    cp_prob = np.zeros(T)
    prec_params = np.array([prec0])
    mean_params = np.array([mean0])
    log_message = np.array([0.0])
    for t in range(T):
        xt = x[t]
        post_prec = prec_params[: t + 1]
        post_var = 1.0 / post_prec + varx
        post_std = np.sqrt(post_var)
        log_preds = norm.logpdf(xt, mean_params[: t + 1], post_std)
        log_growth = log_preds + log_message + log_1mH
        log_cp = logsumexp(log_preds + log_message + log_H)
        new_log_joint = np.append(log_cp, log_growth)
        new_log_joint -= logsumexp(new_log_joint)
        R[t, : t + 2] = np.exp(new_log_joint)
        cp_prob[t] = np.exp(new_log_joint[0])
        new_prec = prec_params + precx
        prec_params = np.append([prec0], new_prec)
        new_mean = (mean_params * prec_params[:-1] + xt * precx) / new_prec
        mean_params = np.append([mean0], new_mean)
        log_message = new_log_joint
    return R[:T], cp_prob

## Load features and run BOCPD

Pick one participant, one condition, one channel, one band. Sequence = 80 reps.

In [ ]:
sub = "sub-01"
cond_idx = 0
ch_idx = 8
band_idx = 2
d = np.load(FEATURES_DIR / f"{sub}.npz", allow_pickle=True)
lbp_test = d["test"]
ch_names = list(d["ch_names"])
bands = list(d["bands"])
x = lbp_test[cond_idx, :, ch_idx, band_idx]
R, cp_prob = bocpd_gaussian(x, hazard=0.02)
print(f"{sub} cond={cond_idx} ch={ch_names[ch_idx]} band={bands[band_idx]}")
print(f"Sequence length: {len(x)}, mean={x.mean():.3f}, std={x.std():.3f}")

## Plot: sequence, run-length posterior, changepoint probability

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
t = np.arange(len(x))
axes[0].plot(t, x, "k-", alpha=0.7)
axes[0].set_ylabel("Log bandpower")
axes[0].set_title(f"{sub} cond {cond_idx} {ch_names[ch_idx]} {bands[band_idx]} — across 80 reps")
axes[0].grid(True, alpha=0.3)
im = axes[1].imshow(R.T[:, : len(x)], aspect="auto", origin="lower", cmap="Blues",
                    extent=[0, len(x), 0, R.shape[1]], vmin=0, vmax=0.3)
axes[1].set_ylabel("Run length")
axes[1].set_title("Run-length posterior p(r_t | x_{1:t})")
plt.colorbar(im, ax=axes[1], label="Prob")
axes[2].plot(t, cp_prob, "C1-", lw=1.5)
axes[2].axhline(0.5, color="gray", ls="--", alpha=0.7)
axes[2].set_ylabel("P(changepoint)")
axes[2].set_xlabel("Repetition index")
axes[2].set_title("Changepoint probability")
axes[2].grid(True, alpha=0.3)
plt.tight_layout()
out_path = ARTIFACTS_FIG / f"bocpd__across_reps__{sub}_cond{cond_idx}_{ch_names[ch_idx]}_{bands[band_idx]}__2026-02-27.png"
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {out_path.name}")

In [ ]:
import pandas as pd
np.random.seed(42)
cond_indices = np.random.choice(200, size=20, replace=False)
summary_rows = []
for sub in ["sub-01", "sub-05", "sub-10"]:
    d = np.load(FEATURES_DIR / f"{sub}.npz", allow_pickle=True)
    lbp = d["test"]
    for c in cond_indices:
        x = lbp[c, :, ch_idx, band_idx]
        _, cp_prob = bocpd_gaussian(x, hazard=0.02)
        summary_rows.append({"participant": sub, "condition": c, "max_cp_prob": float(cp_prob.max())})
summary_df = pd.DataFrame(summary_rows)
table_path = ROOT / "artifacts" / "tables" / "bocpd__across_reps_summary__2026-02-27.csv"
table_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(table_path, index=False)
print(summary_df.groupby("participant")["max_cp_prob"].agg(["mean", "max"]).to_string())

## Run on multiple conditions (sample) and aggregate

In [ ]:
np.random.seed(42)
n_cond_sample = 20
cond_indices = np.random.choice(200, size=n_cond_sample, replace=False)
max_cp = []
for c in cond_indices:
    x = lbp_test[c, :, ch_idx, band_idx]
    _, cp_prob = bocpd_gaussian(x, hazard=0.02)
    max_cp.append(cp_prob.max())
print(f"Max changepoint prob across {n_cond_sample} conditions: mean={np.mean(max_cp):.3f}, max={np.max(max_cp):.3f}")